# Phase 6.4D — Offline Claim & Specification Validation
Replay the six successful frozen extraction responses. Separate exact provenance checks, reusable conservative risk routing, and qualitative development annotations. Zero new API calls. No API key needed. No template labels, automatic clinical confirmation, or DOCX generation.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
ROOT=Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_phase6_specification_validation.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent.agent.phase6_specification_validation import ARCHIVE_NAME, ARCHIVE_SHA, load_frozen, run
import hashlib
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
SOURCE=BASE/'Phase64C_Specification_Pilot_Outputs'
ARCHIVE=SOURCE/ARCHIVE_NAME
if not ARCHIVE.is_file() or hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()!=ARCHIVE_SHA:
    matches=[p for p in SOURCE.glob('*.zip') if hashlib.sha256(p.read_bytes()).hexdigest()==ARCHIVE_SHA]
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected successful frozen archive {ARCHIVE}; matching files: {matches}')
    ARCHIVE=matches[0]
print('Verified successful archive:',ARCHIVE)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT=Path('/content/Phase64D_Validation')/RUN_ID
DEST=BASE/'Phase64D_Validation_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
annotations=json.loads((ROOT/'evaluation/phase64d_development_review.json').read_text(encoding='utf-8'))
requests,claims,old=load_frozen(ARCHIVE)
assert len(requests)==6 and len(claims)==52
print('Frozen requests:',len(requests),'| Claims:',len(claims))

In [ ]:
report=run(ARCHIVE,OUT,annotations=annotations)
assert report['new_llm_calls']==0 and report['original_claims']==52
assert report['development_review_counts']=={'retain_candidate':30,'needs_review':5,'context_only':15,'reject':2}
assert not report['generation_authorized']
print(json.dumps(report,indent=2))
import pandas as pd
from IPython.display import display
review=pd.read_csv(OUT/'Claim_Review.csv')
display(pd.crosstab(review['request_id'],review['development_review']))
display(pd.crosstab(review['original_status'],review['rule_route']))
gaps=pd.read_csv(OUT/'Field_Gap_Review.csv')
display(gaps[['request_id','field','status','review_task']])
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:',sorted(p.name for p in OUT.iterdir()))

## Interpretation
Rule-only specifications are unverified candidates. Development-reviewed specifications use a separate qualitative review of this exact archive only; do not apply those annotations to new studies. Neither is clinical gold. Missing and unresolved fields remain unknown in requiredness. Multiple values need scope checks, not automatic contradiction. No new claims, no template selection or shell generation. Review the gap notes before Phase 6.4E. Share only if authorized to disclose SAP excerpts.